# Enrich Content

Get richer, more detailed understanding of your videos and images for a given context or purpose.
Use this when default extraction is not detailed enough, you need domain-specific metadata, or you want to augment existing understanding with a new lens.

# Install the TwelveLabs Python SDK

In [ ]:
%pip install twelvelabs

In [ ]:
import json
import os

from twelvelabs import TwelveLabs, TextParam
from twelvelabs.types.text_param_format import TextParamFormat_JsonSchema

# Configuration
API_KEY = os.environ.get("TWELVELABS_API_KEY", "YOUR_API_KEY")
STORE_ID = os.environ.get("TWELVELABS_STORE_ID", "your_store_id")  # Replace with your knowledge store ID

client = TwelveLabs(api_key=API_KEY)

## Helper Functions

A utility to extract text content from a Jockey API response.

In [ ]:
def parse_response(response) -> str:
    """Extract text content from a Jockey response.

    Args:
        response: The ResponseObject returned by client.responses.create().

    Returns:
        The text content from the first message output, or an empty string
        if no message content is found.
    """
    for output in response.output:
        if output.type == "message":
            for content in output.content:
                return content.text
    return ""

## Enrichment Schema

Define a JSON schema for enriched video and image analysis. Each enrichment includes an item reference,
the original summary, a deeper enriched analysis, a list of new insights not captured by default
extraction, and domain-specific tags.

In [ ]:
ENRICHMENT_SCHEMA = {
    "type": "object",
    "properties": {
        "enrichments": {
            "type": "array",
            "items": {
                "type": "object",
                "properties": {
                    "item_reference": {"type": "string"},
                    "original_summary": {"type": "string"},
                    "enriched_analysis": {"type": "string"},
                    "new_insights": {
                        "type": "array",
                        "items": {"type": "string"},
                    },
                    "tags": {
                        "type": "array",
                        "items": {"type": "string"},
                    },
                },
            },
        }
    },
}

## Enrich for Brand Strategy

Use the `instructions` field to set Jockey's role as a brand strategist. The enrichment
focuses on brand messaging effectiveness, audience engagement signals, and production quality.
Swap the instructions to enrich for different contexts (see the examples table below).

In [ ]:
response = client.responses.create(
    knowledge_store_id=STORE_ID,
    instructions=(
        "You are a brand strategist. Analyze videos through the lens of "
        "brand perception and marketing effectiveness."
    ),
    input=[
        {
            "type": "message",
            "role": "user",
            "content": (
                "For each video, provide enriched analysis focusing on brand "
                "messaging effectiveness, audience engagement signals, and "
                "production quality."
            ),
        }
    ],
    text=TextParam(
        format=TextParamFormat_JsonSchema(name="content_enrichment", schema_=ENRICHMENT_SCHEMA)
    ),
)

data = json.loads(parse_response(response))

for e in data["enrichments"]:
    print(f"\n{e['item_reference']}")
    print(f"  Analysis: {e['enriched_analysis']}")
    print(f"  Insights: {', '.join(e['new_insights'])}")
    print(f"  Tags: {', '.join(e['tags'])}")

## Example Enrichment Contexts

Swap the `instructions` parameter to enrich for different purposes:

| Context | Instructions |
|---------|--------------|
| Accessibility | "Analyze for accessibility: describe visual elements for screen readers, note caption quality, identify audio-only content" |
| Compliance | "Review for regulatory compliance: identify claims, disclaimers, required disclosures" |
| Education | "Analyze pedagogical effectiveness: identify learning objectives, teaching methods, assessment opportunities" |
| SEO | "Extract SEO metadata: keywords, descriptions, suggested titles, topic clusters" |

## Variations

- **Comparative:** "Enrich by comparing each video to the collection average"
- **Gap analysis:** "What aspects of these videos and images are under-documented?"
- **Audience-specific:** Change instructions to enrich for different target audiences

## Next Steps

- **[Get Corpus Overview](get_corpus_overview.ipynb)** -- understand the full collection first
- **[Search Videos](search_videos.ipynb)** -- find specific moments by description
- **[Extract Entities](extract_entities.ipynb)** -- list all people, places, objects, and concepts
- **[Find Organization Axes](find_organization_axes.ipynb)** -- discover the best categorization strategies

See also:
- [Structured Output Guide](https://docs.twelvelabs.io/v1.3/agents/guides/create-a-response/structured-output) -- more on JSON schema responses
- [Ingestion Config Guide](https://docs.twelvelabs.io/v1.3/agents/guides/create-a-knowledge-store/configure-ingestion) -- configure extraction at index time instead